In [ ]:
import sys
from pathlib import Path

_SRC = Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fraud_detect import config, io, viz, models, ensemble, tuning
from fraud_detect.models import ModelBackend, ModelResult
from fraud_detect.ensemble import EnsembleStrategy, EnsembleConfig

warnings.filterwarnings("ignore")
viz.configure_style()

## 12. Ensemble Methods

Combine LightGBM, XGBoost, and CatBoost via hard/soft voting and stacking.
Ensembles reduce variance and often outperform individual models on
imbalanced data.

> Logic in `fraud_detect.ensemble`.

In [ ]:
train = io.load_train_features()
split = models.make_train_val_split(train)
print(f"Train: {split.X_train.shape}, Val: {split.X_val.shape}")
print(f"Train fraud rate: {split.y_train.mean()*100:.2f}%")
print(f"Val fraud rate  : {split.y_val.mean()*100:.2f}%")

### 12.1 Load Tuned Models

In [ ]:
# Retrain all models with best params from notebook 11
tuned_models = {}
for backend in ModelBackend:
    best_params = tuning.load_best_params(backend, fallback_to_defaults=True)
    result = models.train_model(train, backend=backend, params=best_params)
    tuned_models[backend] = result
    print(f"{backend.value:12s} | Val AUC: {result.val_auc:.5f} | {result.training_time:.1f}s")

### 12.2 Individual Model Predictions

In [ ]:
from sklearn.metrics import roc_auc_score

individual_scores = {}
for backend, result in tuned_models.items():
    preds = result.model.predict_proba(split.X_val)[:, 1]
    auc = roc_auc_score(split.y_val, preds)
    individual_scores[backend.value] = auc

pd.DataFrame([{"Model": k, "Val AUC": v} for k, v in individual_scores.items()])

### 12.3 Hard Voting

In [ ]:
all_backends = list(ModelBackend)
hard_cfg = EnsembleConfig(
    strategy=EnsembleStrategy.HARD_VOTING,
    model_backends=all_backends,
)
hard_ensemble = ensemble.build_voting_ensemble(tuned_models, hard_cfg)
hard_metrics = ensemble.evaluate_ensemble(hard_ensemble, split.X_val, split.y_val)
print(f"Hard Voting — Val AUC: {hard_metrics['auc']:.5f}")
print(f"  F1: {hard_metrics['f1']:.4f}, Precision: {hard_metrics['precision']:.4f}, Recall: {hard_metrics['recall']:.4f}")

### 12.4 Soft Voting

In [ ]:
soft_cfg = EnsembleConfig(
    strategy=EnsembleStrategy.SOFT_VOTING,
    model_backends=all_backends,
)
soft_ensemble = ensemble.build_voting_ensemble(tuned_models, soft_cfg)
soft_metrics = ensemble.evaluate_ensemble(soft_ensemble, split.X_val, split.y_val)
print(f"Soft Voting — Val AUC: {soft_metrics['auc']:.5f}")
print(f"  F1: {soft_metrics['f1']:.4f}, Precision: {soft_metrics['precision']:.4f}, Recall: {soft_metrics['recall']:.4f}")

In [ ]:
# Weighted soft voting — weight proportional to individual AUC
weights = [individual_scores[b.value] for b in all_backends]
weighted_cfg = EnsembleConfig(
    strategy=EnsembleStrategy.SOFT_VOTING,
    model_backends=all_backends,
    weights=weights,
)
weighted_ensemble = ensemble.build_voting_ensemble(tuned_models, weighted_cfg)
weighted_metrics = ensemble.evaluate_ensemble(weighted_ensemble, split.X_val, split.y_val)
print(f"Weighted Soft Voting — Val AUC: {weighted_metrics['auc']:.5f}")
print(f"  Weights: {[round(w, 3) for w in weights]}")

### 12.5 Stacking

In [ ]:
stack_cfg = EnsembleConfig(
    strategy=EnsembleStrategy.STACKING,
    model_backends=all_backends,
)
stack_ensemble = ensemble.build_stacking_ensemble(
    tuned_models, stack_cfg,
    split.X_train, split.y_train,
    split.X_val, split.y_val,
)
stack_metrics = ensemble.evaluate_ensemble(stack_ensemble, split.X_val, split.y_val)
print(f"Stacking — Val AUC: {stack_metrics['auc']:.5f}")
print(f"  F1: {stack_metrics['f1']:.4f}, Precision: {stack_metrics['precision']:.4f}, Recall: {stack_metrics['recall']:.4f}")

### 12.6 Comparison

In [ ]:
# Individual scores only have auc; fill other metrics from ensemble results
comparison = pd.DataFrame([
    {"Method": name, "Val AUC": metrics["auc"], "F1": metrics.get("f1", 0),
     "Precision": metrics.get("precision", 0), "Recall": metrics.get("recall", 0)}
    for name, metrics in [
        ("LightGBM", {"auc": individual_scores["lightgbm"]}),
        ("XGBoost", {"auc": individual_scores["xgboost"]}),
        ("CatBoost", {"auc": individual_scores["catboost"]}),
        ("Hard Voting", hard_metrics),
        ("Soft Voting", soft_metrics),
        ("Weighted Soft", weighted_metrics),
        ("Stacking", stack_metrics),
    ]
])
comparison.sort_values("Val AUC", ascending=False).reset_index(drop=True)

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#4c72b0"]*3 + ["#c44e52"]*4
bars = ax.bar(comparison["Method"], comparison["Val AUC"], color=colors[:len(comparison)])
ax.set_ylabel("Validation AUC")
ax.set_title("Ensemble Methods — Comparison")
ax.set_ylim(0.85, 1.0)
ax.axhline(y=individual_scores["lightgbm"], color="#4c72b0", linestyle=":", alpha=0.5, label="LightGBM ref")

for bar, val in zip(bars, comparison["Val AUC"], strict=True):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{val:.4f}",
            ha="center", va="bottom", fontsize=9)

plt.xticks(rotation=30, ha="right")
plt.tight_layout()
viz.save_figure(fig, config.METADATA_DIR / "ensemble_comparison.png")
plt.show()

### 12.7 Summary

| Method | Val AUC | F1 | Precision | Recall |
|---|---|---|---|---|

Next: notebook 13 — model evaluation & comparison.